# Posterior Predictive

In [ ]:
import numpy as np
import os
import matplotlib.pyplot as plt
import scipy
from signed_inference import *

def typical_elemement(parameters,copy,break_id,sample_size=1,return_avrg=False):
    temp=[]
    for i in range(sample_size):
        x=genrete_binomial_measurements(parameters[1][copy][break_id],parameters[0][copy],12,parameters[2][copy][0][break_id],parameters[3][copy][break_id])
        temp.append(scipy.sparse.csc_matrix(x))
    if return_avrg==True:
        return sum(temp)/len(temp)
    else:
        return temp
def kl(p, q):

    p = np.reshape(p,int(len(p)*(len(p))))
    q = np.reshape(q,int(len(q)*(len(q))))
    return np.sum(np.where(p != 0, p * np.log(p / q), 0))

def discrep(real_mtx,synt_mtx,typ_element):
    #eps=np.ones((len(real_mtx),len(real_mtx)))*0.5**120
    eps=np.zeros((len(real_mtx),len(real_mtx)))
    real_mtx=eps+real_mtx
    synt_mtx=eps+synt_mtx
    eps=np.ones((len(real_mtx),len(real_mtx)))*0.5**120
    typ_element=eps+typ_element
    #a=real_mtx*np.log(real_mtx/typ_element)
    #b=synt_mtx*np.log(synt_mtx/typ_element)
    return kl(real_mtx,typ_element), kl(synt_mtx,typ_element)
def typical_elemement_analytical(parameters,copy,break_id):
    temp=np.zeros((len(parameters[0][copy]),len(parameters[0][copy])))
    
    #x=genrete_binomial_measurements(parameters[1][copy][break_id],parameters[0][copy],12,parameters[2][copy][0][break_id],parameters[3][copy][break_id])
    for i in range(len(parameters[0][copy])):
        for j in range(i):
            if parameters[1][copy][break_id][i]==parameters[1][copy][break_id][j]:
                #print(parameters[1][copy][break_id][i],12*parameters[2][copy][0][break_id][parameters[0][copy][i][j]])
                temp[i][j]=12*parameters[2][copy][0][break_id][parameters[0][copy][i][j]]
                temp[j][i]=temp[i][j]
            else:
                #print(parameters[3][copy][break_id],parameters[2][copy][0][break_id][parameters[0][copy][i][j]])
                temp[i][j]=12*parameters[3][copy][break_id]
                temp[j][i]=temp[i][j]

    x=scipy.sparse.csc_matrix(temp)
    return x
    

In [ ]:
days=['04']
days=[ '02','03', '04', '05', '06']
pvals_dict= {}
vector_of_infs={}
#m0=np.load('observation_mtx2013-12-'+str(num_of_day)+'.npy')
for num_of_day in days:
    m0=np.load('2013-12-'+str(num_of_day)+'.npy')

    mx=np.load(os.getcwd() + str("/diff_days/"+'inf_mtx_')+str(num_of_day)+'.npy')
    part=np.load(os.getcwd() + str("/diff_days/"+'inf_part_')+str(num_of_day)+'.npy',allow_pickle=True)
    q_rate=np.load(os.getcwd() + str("/diff_days/"+'inf_qrt_')+str(num_of_day)+'.npy',allow_pickle=True)
    rate=np.load(os.getcwd() + str("/diff_days/"+'inf_rate_')+str(num_of_day)+'.npy',allow_pickle=True)
    point_est=sum(mx)/len(mx)
    #print(len(m0),len(q_rate[0]),len(rate[0][0]),len(part[0]))
    
    vector_of_infs[num_of_day]=point_est
    params=[mx,part,rate,q_rate]
    t=[]
    pvals=[]
    for break_id in range(len(params[1][0])):

        t=[]
        for copy in range(len(params[0])):

            #a=typical_elemement(params,copy,break_id,150,True)
            a=typical_elemement_analytical(params,copy,break_id)
            synt_inp=genrete_binomial_measurements(params[1][copy][break_id],params[0][copy],12,params[2][copy][0][break_id],params[3][copy][break_id])

            c1,c2=discrep(m0[break_id],synt_inp,a.toarray())
            t.append([c1,c2])
            #print('copy,break_id:',copy,break_id,c1,c2)
        reals=[]
        synths=[]
        p=0
        for x in t:
            reals.append(x[0])
            synths.append(x[1])
            #print(f'real: {reals[-1]}, synthetic: {synths[-1]}')
            if x[0]<x[1]:
                p+=1
        pvals.append(p/len(t))
        print(f"p-value: {p/len(t)} for break_id: {break_id}")
    pvals_dict[num_of_day]=pvals
    plt.scatter(list(range(len(pvals))),pvals,alpha=0.3)
    plt.plot(list(range(len(pvals))),pvals)
    plt.axhline(0.05,color='red',label='0.05 significance level')

    plt.axhline(sum(pvals)/len(pvals),color='green',label='Average Bayesian p-value throughout the day')
    plt.legend()
    plt.show()
    plt.savefig('post_predtrhoughday'+str(num_of_day)+'.pdf')

In [ ]:
pvals_dict

In [ ]:
np.save('pvals',[pvals_dict])

In [ ]:
pvals_dict=np.load('pvals.npy',allow_pickle=True)

In [ ]:
pvals_dict[0]['02']

In [ ]:
pvals_dict=pvals_dict[0]

In [ ]:
colors=['tab:blue','tab:orange','tab:green','tab:red','tab:purple']
#colors=['crimson','magenta','dodgerblue','yellowgreen','red']

t=0
plt.figure(figsize=(12,12))
for key in pvals_dict.keys():
    col=colors[t]
    t+=1
    pvals=pvals_dict[key]
    plt.scatter(list(range(len(pvals))),pvals,alpha=0.8,label='p-values on '+str(key)+' December',color=col)
    plt.plot(list(range(len(pvals))),pvals,alpha=0.2,color=col)
    plt.axhline(np.mean(pvals),color=col,alpha=1,label='average p-value of day '+str(key)+' December')
plt.axhline(0.05,color='black',label='0.05 significance level',linestyle='--')
plt.xlabel('Observation ID',fontsize=18)
plt.ylabel('Bayesian p-value',fontsize=18)
plt.tick_params(axis='both', which='major', labelsize=16)  # Major ticks
#plt.legend()
#plt.legend(bbox_to_anchor=(1.04, 1), loc="upper left",fontsize=15)

#plt.savefig('pvals.pdf')
plt.legend(bbox_to_anchor=(1.04, 1), loc="upper left",fontsize=14)
plt.show()
#plt.savefig('pvals.svg',bbox_inches='tight')

In [ ]:
np.save('bayesian_pvals',pvals_dict)

In [ ]:
#pvals_dict=np.load('pvals.npy',allow_pickle=True)[0]

In [ ]:
p_s=[]
for x in pvals_dict.keys():
    for y in pvals_dict[x]:
        p_s.append(y)

In [ ]:
import datetime
plt.hist(p_s,bins=30,color='salmon')
plt.axvline(0.05,color='black',linestyle='dashed',label='0.05 significance level')
plt.ylim(0,17)
y_min, y_max = plt.ylim()
yticks = np.arange(int(np.floor(y_min / 4)) * 4, int(np.ceil(y_max / 4)) * 4 + 1, 4)
plt.yticks(yticks)
plt.legend()
plt.title('Histogram of Bayesian p-values')
plt.savefig('Hist'+str(datetime.datetime.now())+'.pdf')